# CFLOW Full Two-Stage (Severstal, Colab)

This notebook is hardened for current Colab and this CFLOW repo variant.

Pipeline:
1. Stage-1: CFLOW official script path (`main.py`, `norm-train` only)
2. Stage-2: ResNet50 known-class classifier
3. Integrated two-stage metrics at FPR targets 5/10/20%


In [ ]:
# 0) One-time ABI fix (run then restart runtime)
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy==1.26.4'])
print('Installed numpy==1.26.4. Restart runtime now, then run from cell 1.')


In [ ]:
# 1) Runtime preflight
import sys, numpy as np, torch, torchvision
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
assert np.__version__.startswith('1.26'), 'Need numpy==1.26.x'
print('Preflight OK')


In [ ]:
# 2) Repo setup + deps + compatibility patch
import os, sys, subprocess
from pathlib import Path

FYP_REPO = Path('/content/FYP-code')
if not FYP_REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(FYP_REPO)])
subprocess.check_call(['git', '-C', str(FYP_REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(FYP_REPO), 'reset', '--hard', 'origin/main'])

CFLOW_REPO = Path('/content/cflow-ad')
if not CFLOW_REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/gudovskiy/cflow-ad.git', str(CFLOW_REPO)])
subprocess.check_call(['git', '-C', str(CFLOW_REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(CFLOW_REPO), 'reset', '--hard', 'origin/master'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.7', 'FrEIA==0.2', 'scikit-learn==1.3.2'])

# Pillow>=10 compatibility: replace deprecated ANTIALIAS if present.
patched=[]
for py in CFLOW_REPO.rglob('*.py'):
    txt = py.read_text(errors='ignore')
    new = txt.replace('Image.ANTIALIAS', 'Image.Resampling.LANCZOS')
    if new != txt:
        py.write_text(new)
        patched.append(str(py))

os.chdir(FYP_REPO)
for p in [str(FYP_REPO), str(FYP_REPO/'src'), str(FYP_REPO/'severstral-osr'/'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('FYP repo :', FYP_REPO)
print('CFLOW repo:', CFLOW_REPO)
print('Patched ANTIALIAS in', len(patched), 'files')


In [ ]:
# 3) Drive + dataset checks
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DATA_ROOT = Path('/content/drive/MyDrive/datasets/severstal')
TRAIN_CSV = DATA_ROOT / 'train.csv'
TRAIN_IMAGES = DATA_ROOT / 'train_images'
OUT_ROOT = Path('/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists(), f'Missing: {TRAIN_CSV}'
assert TRAIN_IMAGES.exists() and TRAIN_IMAGES.is_dir(), f'Missing: {TRAIN_IMAGES}'

print('DATA_ROOT:', DATA_ROOT)
print('OUT_ROOT :', OUT_ROOT)


In [ ]:
# 4) Build or reuse split manifest
import csv, json, random
from collections import defaultdict

SPLIT_PATH = OUT_ROOT / 'split_manifest.json'
FORCE_REBUILD_SPLIT = False

if SPLIT_PATH.exists() and not FORCE_REBUILD_SPLIT:
    split = json.loads(SPLIT_PATH.read_text())
    print('Reusing split:', SPLIT_PATH)
else:
    SEED = 42
    KNOWN_CLASSES = ['Class_1', 'Class_2', 'Class_3']
    rng = random.Random(SEED)

    rows_by_image = defaultdict(list)
    with open(TRAIN_CSV, 'r', newline='') as f:
        rd = csv.DictReader(f)
        for r in rd:
            rows_by_image[r['ImageId'].strip()].append(r)

    all_images = [p for p in TRAIN_IMAGES.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]
    normal_paths, single_label = [], []

    for p in all_images:
        rows = rows_by_image.get(p.name, [])
        pos = set()
        for r in rows:
            enc = str(r.get('EncodedPixels','')).strip().lower()
            if enc and enc != 'nan':
                pos.add(int(r['ClassId']))
        if len(pos)==0:
            normal_paths.append(str(p))
        elif len(pos)==1:
            single_label.append((str(p), f"Class_{next(iter(pos))}"))

    known_all = [(p,c) for p,c in single_label if c in KNOWN_CLASSES]
    unknown_all = [(p,c) for p,c in single_label if c not in KNOWN_CLASSES]

    def stratified_split(samples, train_ratio=0.7, val_ratio=0.15, seed=42):
        rr = random.Random(seed)
        by = defaultdict(list)
        for s in samples: by[s[1]].append(s)
        tr,va,te = [],[],[]
        for items in by.values():
            rr.shuffle(items)
            n=len(items); ntr=int(n*train_ratio); nva=int(n*val_ratio)
            tr += items[:ntr]; va += items[ntr:ntr+nva]; te += items[ntr+nva:]
        return tr,va,te

    known_train, known_val, known_test = stratified_split(known_all, seed=SEED)
    rng.shuffle(normal_paths)
    rng.shuffle(unknown_all)

    split = {
        'seed': SEED,
        'known_classes': KNOWN_CLASSES,
        'normal_train': normal_paths[:1200],
        'normal_val': normal_paths[1200:1500],
        'normal_test': normal_paths[1500:1800],
        'known_train': known_train[:900],
        'known_val': known_val[:300],
        'known_test': known_test[:300],
        'unknown_test': unknown_all[:300],
    }

    if min(len(split['normal_train']),len(split['normal_val']),len(split['normal_test']),len(split['known_train']),len(split['known_val']),len(split['known_test']),len(split['unknown_test'])) == 0:
        raise RuntimeError('Split contains empty groups.')

    SPLIT_PATH.write_text(json.dumps(split, indent=2))
    print('Built split:', SPLIT_PATH)

print({k: len(v) for k,v in split.items() if isinstance(v, list)})


In [ ]:
# 5) Build CFLOW datasets in expected path (MVTec-AD), convert to PNG, add required masks
import json, shutil
from pathlib import Path
from PIL import Image
import numpy as np

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())

BASE = Path('/content/cflow-ad/data/MVTec-AD')
CLASS_MAIN = 'bottle'
CLASS_VAL = 'bottle_valonly'
VAL_DEFECT_COUNT = 20  # to avoid ROC-AUC one-class crash in val-only run

MAIN = BASE / CLASS_MAIN
VAL = BASE / CLASS_VAL

for r in [MAIN, VAL]:
    if r.exists():
        shutil.rmtree(r)

for p in [
    MAIN/'train'/'good',
    MAIN/'test'/'good',
    MAIN/'test'/'known_defect',
    MAIN/'test'/'unknown_defect',
    MAIN/'ground_truth'/'known_defect',
    MAIN/'ground_truth'/'unknown_defect',
    VAL/'train'/'good',
    VAL/'test'/'good',
    VAL/'test'/'known_defect',
    VAL/'ground_truth'/'known_defect',
]:
    p.mkdir(parents=True, exist_ok=True)


def write_png(rows, dst, labeled=False):
    n=0
    for i,row in enumerate(rows):
        src = Path(row[0] if labeled else row)
        if not src.exists():
            continue
        out = dst / f'{i:06d}.png'
        try:
            Image.open(src).convert('RGB').save(out, format='PNG')
            n += 1
        except Exception:
            pass
    return n


def write_masks_for_defect_images(test_defect_dir, gt_defect_dir):
    n=0
    for img_path in sorted(test_defect_dir.glob('*.png')):
        m = gt_defect_dir / f'{img_path.stem}_mask.png'
        if m.exists():
            continue
        w,h = Image.open(img_path).size
        Image.fromarray((np.ones((h,w), dtype=np.uint8)*255)).save(m, format='PNG')
        n += 1
    return n

# MAIN dataset
n_m_tr = write_png(split['normal_train'], MAIN/'train'/'good')
n_m_ng = write_png(split['normal_test'], MAIN/'test'/'good')
n_m_kd = write_png(split['known_test'], MAIN/'test'/'known_defect', labeled=True)
n_m_ud = write_png(split['unknown_test'], MAIN/'test'/'unknown_defect', labeled=True)
mk1 = write_masks_for_defect_images(MAIN/'test'/'known_defect', MAIN/'ground_truth'/'known_defect')
mk2 = write_masks_for_defect_images(MAIN/'test'/'unknown_defect', MAIN/'ground_truth'/'unknown_defect')

# VAL dataset (mostly good + tiny defect subset so ROC-AUC is defined)
n_v_tr = write_png(split['normal_train'], VAL/'train'/'good')
n_v_ng = write_png(split['normal_val'], VAL/'test'/'good')
val_def = split['known_test'][:VAL_DEFECT_COUNT]
n_v_kd = write_png(val_def, VAL/'test'/'known_defect', labeled=True)
mk3 = write_masks_for_defect_images(VAL/'test'/'known_defect', VAL/'ground_truth'/'known_defect')

assert n_m_tr > 0 and n_m_ng > 0 and n_m_kd > 0 and n_m_ud > 0, 'MAIN dataset build failed'
assert n_v_tr > 0 and n_v_ng > 0 and n_v_kd > 0, 'VAL dataset build failed'

meta = {
    'class_main': CLASS_MAIN,
    'class_val': CLASS_VAL,
    'val_defect_count': int(n_v_kd),
    'counts': {
        'main_train_good': int(n_m_tr),
        'main_test_good': int(n_m_ng),
        'main_test_known_defect': int(n_m_kd),
        'main_test_unknown_defect': int(n_m_ud),
        'val_train_good': int(n_v_tr),
        'val_test_good': int(n_v_ng),
        'val_test_known_defect': int(n_v_kd),
    }
}
(OUT_ROOT/'cflow_paths.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))


In [ ]:
# 6) Quick sanity run (norm-train only in this repo)
import subprocess, time, json

cfg = json.loads((OUT_ROOT/'cflow_paths.json').read_text())
CLASS_MAIN = cfg['class_main']
cwd = '/content/cflow-ad'
RUN_ID_QUICK = 101

cmd = [
    'python', 'main.py',
    '--dataset', 'mvtec',
    '--class-name', CLASS_MAIN,
    '--action-type', 'norm-train',
    '--run-name', str(RUN_ID_QUICK),
    '--batch-size', '8',
    '--meta-epochs', '1',
    '--sub-epochs', '1',
    '--workers', '2',
    '--input-size', '256',
    '--gpu', '0',
]

start = time.time()
p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
print(p.stdout[-4000:])
print(p.stderr[-4000:])
if p.returncode != 0:
    raise RuntimeError(f'Quick CFLOW run failed with code {p.returncode}')
print('Quick sanity passed in %.1fs' % (time.time()-start))


In [ ]:
# 7) Full stage-1 runs (main then val-only) with metadata
import subprocess, time, json, os, shutil
from pathlib import Path

cfg = json.loads((OUT_ROOT/'cflow_paths.json').read_text())
CLASS_MAIN = cfg['class_main']
CLASS_VAL = cfg['class_val']
cwd = '/content/cflow-ad'

RUN_ID_MAIN = 201
RUN_ID_VAL = 202

meta = {}

# A) main full run
meta['t_main_start'] = time.time()
subprocess.check_call([
    'python', 'main.py',
    '--dataset', 'mvtec',
    '--class-name', CLASS_MAIN,
    '--action-type', 'norm-train',
    '--run-name', str(RUN_ID_MAIN),
    '--batch-size', '16',
    '--meta-epochs', '8',
    '--sub-epochs', '4',
    '--workers', '2',
    '--input-size', '256',
    '--gpu', '0',
], cwd=cwd)
meta['t_main_end'] = time.time()
print('Main run done in %.1fs' % (meta['t_main_end']-meta['t_main_start']))

# B) val-only short run (swap folder into class_main name)
base = Path('/content/cflow-ad/data/MVTec-AD')
main_root = base / CLASS_MAIN
val_root = base / CLASS_VAL
backup = base / f'{CLASS_MAIN}_backup_for_val'
if backup.exists():
    shutil.rmtree(backup)
os.rename(main_root, backup)
os.rename(val_root, main_root)

try:
    meta['t_val_start'] = time.time()
    subprocess.check_call([
        'python', 'main.py',
        '--dataset', 'mvtec',
        '--class-name', CLASS_MAIN,
        '--action-type', 'norm-train',
        '--run-name', str(RUN_ID_VAL),
        '--batch-size', '16',
        '--meta-epochs', '1',
        '--sub-epochs', '1',
        '--workers', '2',
        '--input-size', '256',
        '--gpu', '0',
    ], cwd=cwd)
    meta['t_val_end'] = time.time()
finally:
    os.rename(main_root, val_root)
    os.rename(backup, main_root)

print('Val-only run done in %.1fs' % (meta['t_val_end']-meta['t_val_start']))
(OUT_ROOT/'cflow_run_meta.json').write_text(json.dumps(meta, indent=2))


In [ ]:
# 8) Extract stage-1 score vectors from CFLOW artifacts (hard fail if missing)
import json, pickle
from pathlib import Path
import numpy as np
import pandas as pd

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())
cfg = json.loads((OUT_ROOT/'cflow_paths.json').read_text())
run_meta = json.loads((OUT_ROOT/'cflow_run_meta.json').read_text())

n_main = len(split['normal_test']) + len(split['known_test']) + len(split['unknown_test'])
n_val_good = len(split['normal_val'])
n_val_total = n_val_good + int(cfg.get('val_defect_count', 0))

root = Path('/content/cflow-ad')

# Candidate files updated during/after runs.
min_mtime = float(run_meta['t_main_start']) - 120.0
cands = [p for p in root.rglob('*') if p.is_file() and p.stat().st_mtime >= min_mtime and p.suffix.lower() in {'.npy','.npz','.pkl','.csv'}]
cands = sorted(cands, key=lambda p: p.stat().st_mtime, reverse=True)


def to_numeric_vec(path):
    try:
        if path.suffix == '.npy':
            a = np.load(path, allow_pickle=True)
            a = np.asarray(a).reshape(-1)
            return a.astype(float) if np.issubdtype(a.dtype, np.number) else None
        if path.suffix == '.npz':
            z = np.load(path, allow_pickle=True)
            for k in z.files:
                a = np.asarray(z[k]).reshape(-1)
                if np.issubdtype(a.dtype, np.number):
                    return a.astype(float)
            return None
        if path.suffix == '.pkl':
            with open(path,'rb') as f:
                obj = pickle.load(f)
            if isinstance(obj, (list, tuple, np.ndarray)):
                a=np.asarray(obj).reshape(-1)
                return a.astype(float) if np.issubdtype(a.dtype, np.number) else None
            if isinstance(obj, dict):
                for v in obj.values():
                    if isinstance(v, (list, tuple, np.ndarray)):
                        a=np.asarray(v).reshape(-1)
                        if np.issubdtype(a.dtype, np.number):
                            return a.astype(float)
            return None
        if path.suffix == '.csv':
            df = pd.read_csv(path)
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    return df[col].to_numpy().reshape(-1).astype(float)
            return None
    except Exception:
        return None

main_vec = None
val_vec = None
picked = []
for p in cands:
    v = to_numeric_vec(p)
    if v is None:
        continue
    if main_vec is None and len(v) == n_main:
        main_vec = v
        picked.append(('main', str(p), len(v)))
    elif val_vec is None and len(v) == n_val_total:
        val_vec = v
        picked.append(('val', str(p), len(v)))
    if main_vec is not None and val_vec is not None:
        break

print('Picked:', picked)
if main_vec is None:
    raise RuntimeError(f'Could not find main score vector with length={n_main}')
if val_vec is None:
    raise RuntimeError(f'Could not find val score vector with length={n_val_total}')

n_g = len(split['normal_test'])
n_k = len(split['known_test'])
n_u = len(split['unknown_test'])
if len(main_vec) != (n_g+n_k+n_u):
    raise RuntimeError('Main score length mismatch')

s_nts = main_vec[:n_g]
s_kts = main_vec[n_g:n_g+n_k]
s_uts = main_vec[n_g+n_k:]

# For val run ordering, keep first normal_val segment (dataset built with good first).
s_nva = val_vec[:n_val_good]

np.save(OUT_ROOT/'stage1_scores_normal_val.npy', s_nva)
np.save(OUT_ROOT/'stage1_scores_normal_test.npy', s_nts)
np.save(OUT_ROOT/'stage1_scores_known_test.npy', s_kts)
np.save(OUT_ROOT/'stage1_scores_unknown_test.npy', s_uts)
print('Saved stage1 score files to OUT_ROOT')


In [ ]:
# 9) Stage-2 ResNet50 known-class classifier
import json, numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from src.models.resnet50 import build_resnet50

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())
class_to_idx = {c:i for i,c in enumerate(split['known_classes'])}
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class KnownDS(Dataset):
    def __init__(self, rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        p,c=self.rows[i]
        return tf(Image.open(p).convert('RGB')), class_to_idx[c]

tr=DataLoader(KnownDS(split['known_train']), batch_size=32, shuffle=True, num_workers=2)
va=DataLoader(KnownDS(split['known_val']), batch_size=32, shuffle=False, num_workers=2)

model=build_resnet50(num_classes=len(class_to_idx), pretrained=True).to(device)
opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit=nn.CrossEntropyLoss()

best_va=-1.0; best=None
for ep in range(4):
    model.train()
    for x,y in tr:
        x,y=x.to(device),y.to(device)
        opt.zero_grad(); loss=crit(model(x),y); loss.backward(); opt.step()
    model.eval(); cor=tot=0
    with torch.no_grad():
        for x,y in va:
            p=model(x.to(device)).argmax(1).cpu()
            cor += (p==y).sum().item(); tot += len(y)
    acc=cor/max(1,tot)
    print(f'epoch {ep+1}/4 val_acc={acc:.4f}')
    if acc>best_va:
        best_va=acc; best={k:v.cpu() for k,v in model.state_dict().items()}

model.load_state_dict(best)

def logits(rows):
    class DS(Dataset):
        def __init__(self, rows): self.rows=rows
        def __len__(self): return len(self.rows)
        def __getitem__(self, i):
            r=self.rows[i]
            p=r[0] if isinstance(r,(tuple,list)) else r
            return tf(Image.open(p).convert('RGB'))
    dl=DataLoader(DS(rows), batch_size=32, shuffle=False, num_workers=2)
    out=[]
    model.eval()
    with torch.no_grad():
        for x in dl:
            out.append(model(x.to(device)).cpu().numpy())
    return np.concatenate(out,0)

np.save(OUT_ROOT/'stage2_logits_known_val.npy', logits(split['known_val']))
np.save(OUT_ROOT/'stage2_logits_known_test.npy', logits(split['known_test']))
np.save(OUT_ROOT/'stage2_logits_unknown_test.npy', logits(split['unknown_test']))
print('Saved stage2 logits')


In [ ]:
# 10) Integrated two-stage metrics
import json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())
class_to_idx = {c:i for i,c in enumerate(split['known_classes'])}

def margin(logits):
    p=np.partition(logits, -2, axis=1)
    return p[:,-1]-p[:,-2]

s_nva=np.load(OUT_ROOT/'stage1_scores_normal_val.npy')
s_nts=np.load(OUT_ROOT/'stage1_scores_normal_test.npy')
s_kts=np.load(OUT_ROOT/'stage1_scores_known_test.npy')
s_uts=np.load(OUT_ROOT/'stage1_scores_unknown_test.npy')

lv=np.load(OUT_ROOT/'stage2_logits_known_val.npy')
lk=np.load(OUT_ROOT/'stage2_logits_known_test.npy')
lu=np.load(OUT_ROOT/'stage2_logits_unknown_test.npy')

m_val=margin(lv); m_k=margin(lk); m_u=margin(lu)
pk=lk.argmax(1)
yk=np.array([class_to_idx[c] for _,c in split['known_test']])

auroc=float(roc_auc_score(np.r_[np.zeros(len(s_nts)), np.ones(len(s_kts)+len(s_uts))], np.r_[s_nts, s_kts, s_uts]))
rows=[]
for fpr in [0.05,0.10,0.20]:
    tau=float(np.quantile(s_nva, 1-fpr))
    kappa=float(np.quantile(m_val, fpr))

    n_def=s_nts>tau; k_def=s_kts>tau; u_def=s_uts>tau
    k_unk=m_k<kappa; u_unk=m_u<kappa

    k_succ=k_def & (~k_unk) & (pk==yk)
    u_succ=u_def & u_unk

    rows.append({
        'fpr_target':fpr,
        'AUROC_defect_screening':auroc,
        'TPR_defect@FPR':float(np.mean(np.r_[k_def,u_def])),
        'TPR_unknown@FPR':float(np.mean(u_def)),
        'FPR_known_as_def@FPR':float(np.mean(k_def)),
        'FPR_normal_realized':float(np.mean(n_def)),
        'two_stage_known_success':float(np.mean(k_succ)),
        'two_stage_unknown_success':float(np.mean(u_succ)),
        'stage2_unknown_rate_known_all':float(np.mean(k_unk)),
        'stage2_unknown_rate_unknown_all':float(np.mean(u_unk)),
    })

res=pd.DataFrame(rows)
res.to_csv(OUT_ROOT/'cflow_two_stage_summary.csv', index=False)
print('Saved:', OUT_ROOT/'cflow_two_stage_summary.csv')
display(res)


In [ ]:
# 11) Plot summary
import pandas as pd, matplotlib.pyplot as plt
res=pd.read_csv(OUT_ROOT/'cflow_two_stage_summary.csv')
plt.figure(figsize=(7,4))
plt.plot(res['fpr_target'], res['TPR_defect@FPR'], marker='o', label='TPR_defect')
plt.plot(res['fpr_target'], res['TPR_unknown@FPR'], marker='o', label='TPR_unknown')
plt.plot(res['fpr_target'], res['FPR_normal_realized'], marker='o', label='FPR_normal')
plt.ylim(0,1); plt.xlabel('FPR target'); plt.ylabel('rate'); plt.legend()
plt.tight_layout()
plt.savefig(OUT_ROOT/'plot_cflow_two_stage_rates.png', dpi=140)
plt.show()
print('Saved:', OUT_ROOT/'plot_cflow_two_stage_rates.png')


## Run Order

After runtime restart:
1. Run cells 1 -> 11 in order.
2. Do not skip cell 2 (repo reset + compatibility patch).
3. If cell 8 fails, keep its printed `Picked` candidates and share them for one targeted parser tweak.
